In [1]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.models.schemas import TimeframeDetection, InputLanguage
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector


load_dotenv()

/home/lamossta/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Downloading and creating SPARQL endpoint for the NKOD metadata

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.download_catalog_metadata()
#nkod_data_processor.create_metadata_csv(graph_db)
#nkod_data_processor.create_themes_csv(graph_db)
#nkod_data_processor.create_metadata_sql(sq_lite)
#nkod_data_processor.create_themes_sql(sq_lite)

## Indexing the keywords, titles and descriptions from the NKOD metadata (TODO: matching a dataset)

In [3]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-small")
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)
#nkod_data_processor.index_catalog_themes(sq_lite, openai_embeddings, chroma_db)
#nkod_data_processor.index_catalog_metadata(sq_lite, openai_embeddings, chroma_db, verbose=True)
#chroma_db.delete_collection("nkod_themes_definitions_cs")
print(chroma_db.list_collections())

['nkod_themes_definitions_en', 'nkod_keywords_en', 'nkod_descriptions_en', 'nkod_keywords_cs', 'nkod_themes_labels_en', 'nkod_descriptions_cs', 'nkod_themes_labels_cs', 'nkod_titles_en', 'nkod_themes_definitions_cs', 'nkod_titles_cs']


## Language detection, Timeframe detection and Query matching

In [4]:
input_query = ""

model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)

#timeframe_detection = TimeframeDetector().detect(input_query, openai_llm, model_name)
#language_detection = LanguageDetector().detect(input_query, openai_llm, model_name)
#print(f"Language: {language_detection}, Timeframe: {timeframe_detection}")

k = 50
nkod_query_matcher = NkodQueryMatcher(input_query)
#print(nkod_query_matcher.high_k_intersection(k, chroma_db, nkod_data_processor, language_detection.language, openai_embeddings))
#print(nkod_query_matcher.get_matching_titles(5, chroma_db, nkod_data_processor, language_detection.language.value, openai_embeddings))
results = nkod_query_matcher.run_best_k_search(chroma_db, nkod_data_processor, openai_embeddings, InputLanguage.CZECH)
print(results)
nkod_query_matcher.evaluate_on_ofn_dataset(chroma_db, nkod_data_processor, "cs", openai_embeddings)
nkod_query_matcher.query = "official bulletin board ricany"
nkod_query_matcher.get_matching_titles(50, chroma_db, nkod_data_processor, "en", openai_embeddings)

[{'number_of_queries': 25, 'total_matches_intersections': 0, 'mean_matches_intersections': 0.0, 'total_matches_titles': 0, 'mean_matches_titles': 0.0}]
Query 1/4
{'query': 'Jaké události se budou odehrávat na Vyžlovce', 'language': 'cs', 'timeframe': 'None', 'dataset_uris': ['https://data.gov.cz/zdroj/datové-sady/00235938/1355924022'], 'theme_names': ['GOVE'], 'theme_labels': ['Vláda a veřejný sektor'], 'theme_definitions': ['téma datového souboru týkající se oblastí správy a veřejného sektoru, přičemž správou se rozumí systém nebo skupina jednotlivců řídící organizovanou komunitu, obvykle stát, a veřejný sektor se skládá ze segmentu hospodářství zahrnujícího veřejné služby a veřejné podniky, který může být řízen ústředními, regionálními nebo místními orgány']}


Number of requested results 20 is greater than number of elements in index 13, updating n_results = 13
Number of requested results 20 is greater than number of elements in index 14, updating n_results = 14


In definitions: False
Definitions similarity matching: [(0.6746220278184134, 'téma datového souboru týkající se oblasti mezinárodních otázek, která se vztahuje na významná témata nebo problémy, o nichž se vede debata nebo diskuse s účastníky z nejméně dvou různých zemí'), (0.6955292137328372, 'téma datového souboru týkající se oblasti zemědělství, jež zahrnuje pěstování rostlin a chov hospodářských zvířat, oblasti rybolovu, který se zaměřuje na výlov volně žijících ryb nebo ryb ve farmovém chovu, oblasti lesnictví, které se vztahuje na obhospodařování a ochranu lesů a zalesněných ploch, a oblasti potravin, jež zahrnuje látky poskytující nutriční podporu živým organismům'), (0.697909290582809, 'téma datového souboru týkající se oblasti životního prostředí, definované jako interakce všech živých druhů, klimatu, počasí a přírodních zdrojů, jež ovlivňuje přežití lidstva a hospodářskou činnost'), (0.7103023701029679, 'téma datového souboru týkající se oblasti vzdělávání, které zprostředková

Number of requested results 20 is greater than number of elements in index 13, updating n_results = 13
Number of requested results 20 is greater than number of elements in index 14, updating n_results = 14


In definitions: False
Definitions similarity matching: [(0.6072038882497617, 'téma datového souboru týkající se oblasti hospodářské činnosti, jež zahrnuje výrobu a distribuci zboží a služeb, obchodování s nimi a jejich spotřebu, a oblasti finanční činnosti, která se zaměřuje na správu peněz na úrovni jednotlivců, podniků nebo veřejné správy'), (0.6314570793900033, 'téma datového souboru týkající se oblastí správy a veřejného sektoru, přičemž správou se rozumí systém nebo skupina jednotlivců řídící organizovanou komunitu, obvykle stát, a veřejný sektor se skládá ze segmentu hospodářství zahrnujícího veřejné služby a veřejné podniky, který může být řízen ústředními, regionálními nebo místními orgány'), (0.6612873530677117, 'téma datového souboru týkající se oblasti zdraví, která zahrnuje zdravotní obtíže, onemocnění, léčbu, zdravotnické služby a zdravotní politiky'), (0.6669533032335555, 'téma datového souboru týkající se oblasti mezinárodních otázek, která se vztahuje na významná témata

Number of requested results 20 is greater than number of elements in index 13, updating n_results = 13
Number of requested results 20 is greater than number of elements in index 14, updating n_results = 14


In definitions: False
Definitions similarity matching: [(0.5535668902982089, 'téma datového souboru týkající se oblasti vzdělávání, které zprostředkovává učení a získávání znalostí, dovedností, hodnot, přesvědčení a zvyklostí, oblasti kultury, jež zahrnuje společenské chování, normy, znalosti, přesvědčení, umění, právní předpisy, obyčeje, schopnosti a zvyklosti existující v lidské společnosti, a oblasti sportu, jenž zahrnuje závodní fyzické aktivity nebo hry, které zlepšují fyzické schopnosti a dovednosti účastníků, a to jak pro jejich potěšení, tak pro zábavu diváků'), (0.578851817076326, 'téma datového souboru týkající se oblasti dopravy, jež zahrnuje přepravu lidí, zvířat a zboží z jednoho místa do druhého za použití různých prostředků, jako je letecká, pozemní (železniční a silniční), vodní, lanová, potrubní a kosmická doprava'), (0.6078076841893878, 'téma datového souboru týkající se oblasti hospodářské činnosti, jež zahrnuje výrobu a distribuci zboží a služeb, obchodování s nimi 

Number of requested results 20 is greater than number of elements in index 13, updating n_results = 13
Number of requested results 20 is greater than number of elements in index 14, updating n_results = 14


In definitions: False
Definitions similarity matching: [(0.6573337237179351, 'téma datového souboru týkající se oblasti mezinárodních otázek, která se vztahuje na významná témata nebo problémy, o nichž se vede debata nebo diskuse s účastníky z nejméně dvou různých zemí'), (0.7323682791129629, 'téma datového souboru týkající se oblasti dopravy, jež zahrnuje přepravu lidí, zvířat a zboží z jednoho místa do druhého za použití různých prostředků, jako je letecká, pozemní (železniční a silniční), vodní, lanová, potrubní a kosmická doprava'), (0.7369723207084313, 'téma datového souboru týkající se oblasti zdraví, která zahrnuje zdravotní obtíže, onemocnění, léčbu, zdravotnické služby a zdravotní politiky'), (0.7387538823039845, 'téma datového souboru týkající se oblastí spravedlnosti, právního systému a veřejné bezpečnosti, kde spravedlnost znamená procesní spravedlnost při uplatňování práva, právní systém zahrnuje různé právní rámce, zahrnující občanské, zvykové, statutární a náboženské prá

['https://data.gov.cz/zdroj/datové-sady/00299308/1570615018',
 'https://data.gov.cz/zdroj/datové-sady/17651921/41d2957c0c7d2fb9e22d9cd2324efcb5',
 'https://data.gov.cz/zdroj/datové-sady/70890692/0f1a6a154f04ed242343d96defbdc978',
 'https://data.gov.cz/zdroj/datové-sady/70890692/228955827a9387cb4ecea905ca80f531',
 'https://data.gov.cz/zdroj/datové-sady/70890692/47d21f513102b6d175368893f83b7049',
 'https://data.gov.cz/zdroj/datové-sady/48135097/71e80600130c39860e7dfb58a5017e15',
 'https://data.gov.cz/zdroj/datové-sady/48135097/fc5fff0f5887dcaf86c48cb4cecdfcca',
 'https://data.gov.cz/zdroj/datové-sady/48135097/57dec402bccc035fb3ff4425024cba85',
 'https://data.gov.cz/zdroj/datové-sady/48135097/0af1ff01268d76ae2c6efdedce8bcd40',
 'https://data.gov.cz/zdroj/datové-sady/48135097/f93352fba0a65563eae184761413d802',
 'https://data.gov.cz/zdroj/datové-sady/48135097/53facedb478f2c3865b6110026eec619',
 'https://data.gov.cz/zdroj/datové-sady/48135097/3af3cb19061788bb67213e12fa52e525',
 'https://data